In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets scikit-learn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 24.6 MB/s eta 0:00:00


In [ ]:
# 3. Imports
import os
import json
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset
import datasets
datasets.config.TORCHVISION_AVAILABLE = False   # fixes VideoReader ImportError

In [ ]:
# 4. Paths (update DRIVE_FOLDER to your actual mounted path)
DRIVE_FOLDER = "/content/drive/MyDrive/FIRE-2026"  # <-- update this
TRAIN_PATH = f"{DRIVE_FOLDER}/train_subtask1.json"
DEV_PATH = f"{DRIVE_FOLDER}/dev_subtask1.json"
OUTPUT_DIR = "/content/drive/MyDrive/FIRE-2026/FIRE-2026-HPT"
BEST_MODEL_DIR = f"{OUTPUT_DIR}/best_hp_model"

In [ ]:
# ============================================
# 5. DATA PREP
# ============================================

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

label2id = {"SUPPORTS": 0, "REFUTES": 1}
id2label = {0: "SUPPORTS", 1: "REFUTES"}

def to_hf_dataset(data):
    return Dataset.from_dict({
        "claim": [x["Text"] for x in data],
        "evidence": [x["Evidence"] for x in data],
        "label": [label2id[x["Label"]] for x in data],
    })

MODEL_NAME = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["claim"],
        examples["evidence"],
        truncation=True,
        max_length=192,
        padding="max_length",
    )

train_dataset = to_hf_dataset(train_data).map(tokenize_fn, batched=True)
dev_dataset = to_hf_dataset(dev_data).map(tokenize_fn, batched=True)

train_dataset = train_dataset.remove_columns(["claim", "evidence"])
dev_dataset = dev_dataset.remove_columns(["claim", "evidence"])

train_dataset.set_format("torch")
dev_dataset.set_format("torch")

assert "input_ids" in train_dataset.column_names, "Tokenization failed!"
print("✅ Columns:", train_dataset.column_names)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/4536 [00:00<?, ? examples/s]

Map:   0%|          | 0/1134 [00:00<?, ? examples/s]

✅ Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ============================================
# 6. Class weights (auto-computed from data)
# ============================================

labels = [x["label"].item() for x in train_dataset]
class_counts = np.bincount(labels)
class_weights = torch.tensor(
    [len(labels) / (2 * c) for c in class_counts], dtype=torch.float
)
print("Class weights:", class_weights)

Class weights: tensor([0.7461, 1.5160])


In [ ]:
# ============================================
# 7. Model init function (required for hyperparameter_search —
#    reinitializes a fresh model for every trial)
# ============================================

def model_init(trial=None):
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
    )

In [ ]:
# ============================================
# 8. Custom Trainer with weighted loss
# ============================================

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================
# 9. Metrics
# ============================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")

    precision, recall, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], zero_division=0
    )

    return {
        "accuracy": acc,
        "f1": f1_macro,
        "precision_supports": precision[0],
        "recall_supports": recall[0],
        "f1_supports": f1_per_class[0],
        "precision_refutes": precision[1],
        "recall_refutes": recall[1],
        "f1_refutes": f1_per_class[1],
    }

In [ ]:
# ============================================
# 10. Base Training Arguments (values here get overridden per-trial by hp_space)
# ============================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,   # keeps trial logs cleaner
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# ============================================
# 11. Trainer (model=None, uses model_init for hyperparameter_search)
# ============================================

trainer = WeightedTrainer(
    model=None,
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
# ============================================
# 12. Optuna search space
# ============================================

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 4, 16),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "max_grad_norm": trial.suggest_float("max_grad_norm", 0.5, 2.0),
    }

# Objective: maximize macro F1 on dev set
def compute_objective(metrics):
    return metrics["eval_f1"]

In [ ]:
# ============================================
# 13. Run Hyperparameter Search (with persistent storage)
# ============================================

storage_path = f"sqlite:////content/drive/MyDrive/FIRE-2026/optuna_study.db"

best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    compute_objective=compute_objective,
    n_trials=15,
    study_name="muril_hp_search",
    storage=storage_path,
    load_if_exists=True,
)

print("✅ Best trial:", best_trial)

[I 2026-07-22 04:23:14,852] A new study created in RDB with name: muril_hp_search


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

{'loss': '1.386', 'grad_norm': '0.138', 'learning_rate': '7.733e-06', 'epoch': '0.7042'}
{'eval_loss': '0.6931', 'eval_accuracy': '0.6508', 'eval_f1': '0.5099', 'eval_precision_supports': '0.6723', 'eval_recall_supports': '0.9082', 'eval_f1_supports': '0.7727', 'eval_precision_refutes': '0.4887', 'eval_recall_refutes': '0.1654', 'eval_f1_refutes': '0.2471', 'eval_runtime': '3.567', 'eval_samples_per_second': '317.9', 'eval_steps_per_second': '39.81', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.385', 'grad_norm': '0.4334', 'learning_rate': '1.562e-05', 'epoch': '1.408'}
{'eval_loss': '0.6719', 'eval_accuracy': '0.7028', 'eval_f1': '0.6608', 'eval_precision_supports': '0.7551', 'eval_recall_supports': '0.807', 'eval_f1_supports': '0.7802', 'eval_precision_refutes': '0.5819', 'eval_recall_refutes': '0.5064', 'eval_f1_refutes': '0.5415', 'eval_runtime': '3.589', 'eval_samples_per_second': '316', 'eval_steps_per_second': '39.56', 'epoch': '2'}


... (166 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

{'loss': '1.387', 'grad_norm': '0.07647', 'learning_rate': '3.348e-06', 'epoch': '0.3521'}
{'loss': '1.386', 'grad_norm': '0.1005', 'learning_rate': '6.764e-06', 'epoch': '0.7042'}


[I 2026-07-22 05:48:28,617] Trial 14 pruned. 


{'eval_loss': '0.6917', 'eval_accuracy': '0.6649', 'eval_f1': '0.4322', 'eval_precision_supports': '0.6613', 'eval_recall_supports': '0.9987', 'eval_f1_supports': '0.7957', 'eval_precision_refutes': '0.9333', 'eval_recall_refutes': '0.03562', 'eval_f1_refutes': '0.06863', 'eval_runtime': '3.842', 'eval_samples_per_second': '295.2', 'eval_steps_per_second': '36.96', 'epoch': '1'}
✅ Best trial: BestRun(run_id='0', objective=0.801407937856536, hyperparameters={'learning_rate': 1.7991465194475932e-05, 'per_device_train_batch_size': 32, 'num_train_epochs': 16, 'weight_decay': 0.0764751706565559, 'warmup_ratio': 0.026509480810828625, 'max_grad_norm': 1.2004964752527867}, run_summary=None)


In [ ]:
# ============================================
# 14. Retrain final model using best hyperparameters
# ============================================

for k, v in best_trial.hyperparameters.items():
    setattr(training_args, k, v)

final_trainer = WeightedTrainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

{'loss': '1.386', 'grad_norm': '0.138', 'learning_rate': '7.733e-06', 'epoch': '0.7042'}
{'eval_loss': '0.6931', 'eval_accuracy': '0.6517', 'eval_f1': '0.5118', 'eval_precision_supports': '0.673', 'eval_recall_supports': '0.9082', 'eval_f1_supports': '0.7731', 'eval_precision_refutes': '0.4925', 'eval_recall_refutes': '0.1679', 'eval_f1_refutes': '0.2505', 'eval_runtime': '3.913', 'eval_samples_per_second': '289.8', 'eval_steps_per_second': '36.29', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (28 outputs left out)

{'loss': '0.4238', 'grad_norm': '9.649', 'learning_rate': '6.514e-07', 'epoch': '15.49'}
{'eval_loss': '0.5616', 'eval_accuracy': '0.8254', 'eval_f1': '0.8068', 'eval_precision_supports': '0.8644', 'eval_recall_supports': '0.8691', 'eval_f1_supports': '0.8668', 'eval_precision_refutes': '0.7506', 'eval_recall_refutes': '0.743', 'eval_f1_refutes': '0.7468', 'eval_runtime': '3.938', 'eval_samples_per_second': '287.9', 'eval_steps_per_second': '36.06', 'epoch': '16'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1254', 'train_samples_per_second': '57.87', 'train_steps_per_second': '0.906', 'train_loss': '0.7711', 'epoch': '16'}


TrainOutput(global_step=1136, training_loss=0.7710804821739734, metrics={'train_runtime': 1254.1892, 'train_samples_per_second': 57.867, 'train_steps_per_second': 0.906, 'train_loss': 0.7710804821739734, 'epoch': 16.0})

In [ ]:
# ============================================
# 15. Save best model to Drive
# ============================================

final_trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print("✅ Best model saved at:", BEST_MODEL_DIR)

metrics = final_trainer.evaluate()
print(metrics)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved at: /content/drive/MyDrive/FIRE-2026/FIRE-2026-HPT/best_hp_model
{'eval_loss': '0.5189', 'eval_accuracy': '0.8325', 'eval_f1': '0.812', 'eval_precision_supports': '0.8592', 'eval_recall_supports': '0.8893', 'eval_f1_supports': '0.874', 'eval_precision_refutes': '0.7766', 'eval_recall_refutes': '0.7252', 'eval_f1_refutes': '0.75', 'eval_runtime': '3.441', 'eval_samples_per_second': '329.5', 'eval_steps_per_second': '41.26', 'epoch': '16'}
{'eval_loss': 0.5188919305801392, 'eval_accuracy': 0.8324514991181657, 'eval_f1': 0.8120026525198939, 'eval_precision_supports': 0.8591916558018253, 'eval_recall_supports': 0.8893387314439946, 'eval_f1_supports': 0.8740053050397878, 'eval_precision_refutes': 0.776566757493188, 'eval_recall_refutes': 0.7251908396946565, 'eval_f1_refutes': 0.75, 'eval_runtime': 3.4413, 'eval_samples_per_second': 329.528, 'eval_steps_per_second': 41.264, 'epoch': 16.0}
